# ODD Analysis Workflow - Interactive Version

This notebook implements the complete multi-agent ODD/COD analysis pipeline interactively.

**What this notebook does:**
- Analyzes robot sensor data (camera, LiDAR, IMU) across time windows
- Detects operational design domain (ODD) violations
- Classifies current operating domain (COD) vs design specifications
- Generates comprehensive reports with collision risk assessment

**Pipeline stages:**
1. ODD Specification (natural language → formal spec)
2. Perception Analysis (camera + LiDAR occupancy)
3. Motion Analysis (IMU sensors)
4. Collision Risk Assessment (multimodal fusion)
5. COD Classification (current conditions)
6. ODD Compliance Check (violations detection)
7. Report Generation

## 1. Imports and Environment Setup

In [ ]:
import asyncio
import json
import math
import os
from pathlib import Path
from typing import Any, Dict, List, Optional

from dotenv import load_dotenv
from google import genai
from google.genai import types

from google.adk.agents import Agent, SequentialAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools import FunctionTool
from google.adk.tools.tool_context import ToolContext

# Load environment variables
load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

if not GOOGLE_API_KEY:
    raise SystemExit("❌ GOOGLE_API_KEY not found. Set it in your environment or .env file.")

print("✓ All imports successful")
print(f"✓ Google API key configured")

## 2. Configuration

In [ ]:
# Project paths
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "processed" / "runs"

# Choose your scenario
# Options: "demo_run", "run_001", "sim_run_test", "sim_run_new", "test_run"
SCENARIO = "sim_run_test"
SCENARIO_PATH = DATA_DIR / SCENARIO

# Model assignments per agent (optimized for cost/performance)
GEMINI_MODEL_PERCEPTION = "gemini-2.5-pro"      # Vision analysis needs accuracy
GEMINI_MODEL_MOTION = "gemini-2.5-pro"          # Data aggregation needs structure
GEMINI_MODEL_COLLISION = "gemini-2.5-pro"       # Complex multimodal fusion
GEMINI_MODEL_ODD_SPEC = "gemini-2.0-flash-lite" # JSON synthesis only
GEMINI_MODEL_COD = "gemini-2.0-flash-lite"      # Comparison logic
GEMINI_MODEL_REPORT = "gemini-2.5-pro"          # High-quality report generation

# Initialize Gemini client
GENAI_CLIENT = genai.Client(api_key=GOOGLE_API_KEY)

# ODD specification (natural language)
NL_ODD_DESCRIPTION = (
    "A quadruped robot designed for indoor office environments. "
    "Operates on smooth, flat floors with adequate lighting (bright or dim). "
    "Maximum speed 1.5 m/s. Designed for environments with moderate obstacle "
    "density and good traversability. Requires low collision risk conditions. "
    "Not designed for: outdoor environments, stairs, rough terrain, "
    "dark/low-light areas, or high-density obstacle fields."
)

print(f"Scenario: {SCENARIO}")
print(f"Data path: {SCENARIO_PATH}")
print(f"ODD: {NL_ODD_DESCRIPTION[:80]}...")

## 3. Utility Functions

In [ ]:
def _build_image_path(prefix: str, window_id: str) -> Path:
    """Build path to image file for a window."""
    scenario_name = SCENARIO_PATH.name
    filename = f"{prefix}_{scenario_name}_w{window_id}.png"
    return SCENARIO_PATH / filename


def _ensure_image_bytes(path: Path) -> bytes:
    """Load image bytes, raising error if missing."""
    if not path.exists():
        raise FileNotFoundError(f"Missing image: {path}")
    return path.read_bytes()


def _extract_json_block(text: str) -> Dict[str, Any]:
    """Extract JSON object from text that may contain markdown code blocks."""
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = "\n".join(line for line in cleaned.splitlines()
                            if not line.strip().startswith("```"))
    start = cleaned.find("{")
    end = cleaned.rfind("}")
    if start == -1 or end == -1:
        raise ValueError(f"No JSON object found in response: {text}")
    return json.loads(cleaned[start:end + 1])

print("✓ Utility functions defined")

## 4. Agent Tool Functions

These functions are called by agents to analyze sensor data.

In [ ]:
async def list_windows_tool() -> Dict[str, Any]:
    """Tool: list available window IDs for the scenario."""
    import pandas as pd

    if not SCENARIO_PATH.exists():
        return {"status": "error", "message": "Scenario directory not found"}

    index_files = sorted(SCENARIO_PATH.glob("index_*.csv"))
    if not index_files:
        return {"status": "error", "message": "No index CSV found"}

    index_df = pd.read_csv(index_files[0])
    scenario_name = SCENARIO_PATH.name
    windows: List[str] = []

    for _, row in index_df.iterrows():
        window_id = str(row["window_id"]).zfill(3)
        motion_file = SCENARIO_PATH / f"motion_{scenario_name}_w{window_id}.json"
        if motion_file.exists():
            windows.append(window_id)

    return {
        "status": "success",
        "windows": windows,
        "count": len(windows),
    }


async def analyze_window_perception_tool(window_id: str, tool_context: ToolContext) -> Dict[str, Any]:
    """Tool: multimodal perception analysis (camera + BEV)."""
    try:
        camera_path = _build_image_path("cam", window_id)
        bev_path = _build_image_path("bev_occupancy", window_id)

        camera_bytes = _ensure_image_bytes(camera_path)
        bev_bytes = _ensure_image_bytes(bev_path)

        prompt = f"""
        You are a perception expert analyzing synchronized robot sensors for window {window_id}.
        You will receive two images:
        - Image A: RGB camera frame from the robot's forward camera.
        - Image B: LiDAR bird's-eye occupancy map where bright pixels indicate obstacles.

        Provide a JSON object with this EXACT schema:
        {{
          "window_id": "{window_id}",
          "camera_summary": "concise natural-language observation",
          "bev_summary": "concise LiDAR occupancy observation",
          "lighting_class": "bright|dim|dark",
          "visibility_score": 0.0-1.0,
          "terrain_roughness_class": "smooth|moderate|rough|very_rough",
          "occupancy_ratio": 0.0-1.0,
          "obstacle_density": 0.0-1.0,
          "traversability_score": 0.0-1.0,
          "humans_detected": true|false,
          "environmental_constraints": ["list", "of", "observed", "constraints"]
        }}

        No explanations, just the JSON.
        """

        response = GENAI_CLIENT.models.generate_content(
            model=GEMINI_MODEL_PERCEPTION,
            contents=[
                types.Part(text=prompt.strip()),
                types.Part(text="Image A (camera):"),
                types.Part.from_bytes(data=camera_bytes, mime_type="image/png"),
                types.Part(text="Image B (LiDAR BEV occupancy):"),
                types.Part.from_bytes(data=bev_bytes, mime_type="image/png"),
            ],
        )

        data = _extract_json_block(response.text or "")
        data["window_id"] = window_id
        return data

    except Exception as err:
        return {"status": "error", "window_id": window_id, "message": str(err)}

print("✓ Perception tool functions defined")

In [ ]:
async def analyze_motion_tool(window_id: str, tool_context: ToolContext) -> Dict[str, Any]:
    """Tool: analyze raw IMU motion sensor data."""
    try:
        scenario_name = SCENARIO_PATH.name
        motion_file = SCENARIO_PATH / f"motion_{scenario_name}_w{window_id}.json"

        if not motion_file.exists():
            return {"status": "error", "window_id": window_id, "message": "Motion file not found"}

        with open(motion_file, 'r') as f:
            motion_data = json.load(f)

        # Calculate summary statistics
        accel_x = motion_data["accel_x"]
        accel_y = motion_data["accel_y"]
        gyro_z = motion_data["gyro_z"]
        roll = motion_data["roll"]
        pitch = motion_data["pitch"]

        horiz_accel = [math.sqrt(ax**2 + ay**2) for ax, ay in zip(accel_x, accel_y)]
        peak_horiz_accel = max(horiz_accel) if horiz_accel else 0.0
        avg_horiz_accel = sum(horiz_accel) / len(horiz_accel) if horiz_accel else 0.0
        peak_gyro_z = max(abs(gz) for gz in gyro_z) if gyro_z else 0.0
        avg_gyro_z = sum(abs(gz) for gz in gyro_z) / len(gyro_z) if gyro_z else 0.0
        max_roll = max(abs(r) for r in roll) if roll else 0.0
        max_pitch = max(abs(p) for p in pitch) if pitch else 0.0

        prompt = f"""You are a robotics motion analyst for window {window_id}.

IMU ACCELEROMETER DATA (gravity-compensated, body frame):
- Horizontal acceleration samples (sqrt(accel_x² + accel_y²)): {len(horiz_accel)} samples
- Peak horizontal accel: {peak_horiz_accel:.4f} m/s²
- Average horizontal accel: {avg_horiz_accel:.4f} m/s²
- Sample values: {horiz_accel[:10]} (first 10 of {len(horiz_accel)})

IMU GYROSCOPE DATA:
- Peak angular velocity (|gyro_z|): {peak_gyro_z:.4f} rad/s
- Average angular velocity: {avg_gyro_z:.4f} rad/s
- Sample values: {gyro_z[:10]} (first 10 of {len(gyro_z)})

PLATFORM ORIENTATION:
- Max roll: {max_roll:.1f}°
- Max pitch: {max_pitch:.1f}°

MOTION DETECTION GUIDANCE:
- Horizontal accel > 0.05 m/s² indicates translation
- Horizontal accel > 0.5 m/s² indicates strong acceleration
- Angular velocity > 0.1 rad/s indicates rotation
- Roll/pitch > 15° indicates platform instability

TASK: Analyze and provide JSON with this EXACT schema:
{{
  "window_id": "{window_id}",
  "motion_detected": true|false,
  "motion_type": "stationary|rotation|translation|complex",
  "peak_horizontal_accel_mps2": <float>,
  "peak_angular_velocity_radps": <float>,
  "platform_stability": "stable|unstable",
  "max_tilt_deg": <float>,
  "motion_confidence": 0.0-1.0,
  "evidence": "brief explanation"
}}

No explanations outside the JSON."""

        response = GENAI_CLIENT.models.generate_content(
            model=GEMINI_MODEL_MOTION,
            contents=[types.Part(text=prompt.strip())],
        )

        data = _extract_json_block(response.text or "")
        data["window_id"] = window_id
        return data

    except Exception as err:
        return {"status": "error", "window_id": window_id, "message": str(err)}

print("✓ Motion tool function defined")

In [ ]:
async def analyze_collision_risk_tool(window_id: str, tool_context: ToolContext) -> Dict[str, Any]:
    """Tool: multimodal collision risk assessment (motion + camera + BEV)."""
    try:
        scenario_name = SCENARIO_PATH.name

        motion_file = SCENARIO_PATH / f"motion_{scenario_name}_w{window_id}.json"
        if not motion_file.exists():
            return {"status": "error", "window_id": window_id, "message": "Motion file not found"}

        with open(motion_file, 'r') as f:
            motion_data = json.load(f)

        camera_path = SCENARIO_PATH / f"cam_{scenario_name}_w{window_id}.png"
        bev_path = SCENARIO_PATH / f"bev_occupancy_{scenario_name}_w{window_id}.png"

        if not camera_path.exists() or not bev_path.exists():
            return {"status": "error", "window_id": window_id, "message": "Images not found"}

        camera_bytes = camera_path.read_bytes()
        bev_bytes = bev_path.read_bytes()

        # Calculate motion metrics
        accel_x = motion_data["accel_x"]
        accel_y = motion_data["accel_y"]
        gyro_z = motion_data["gyro_z"]
        roll = motion_data["roll"]
        pitch = motion_data["pitch"]

        horiz_accel = [math.sqrt(ax**2 + ay**2) for ax, ay in zip(accel_x, accel_y)]
        peak_horiz_accel = max(horiz_accel) if horiz_accel else 0.0
        peak_gyro_z = max(abs(gz) for gz in gyro_z) if gyro_z else 0.0
        max_tilt = max(max(abs(r) for r in roll) if roll else 0.0,
                       max(abs(p) for p in pitch) if pitch else 0.0)

        motion_summary = {
            "peak_horizontal_accel_mps2": round(peak_horiz_accel, 3),
            "peak_angular_velocity_radps": round(peak_gyro_z, 3),
            "max_tilt_deg": round(max_tilt, 1),
        }

        prompt = f"""You are a collision risk assessment expert for window {window_id}.

MOTION DATA:
{json.dumps(motion_summary, indent=2)}

VISUAL DATA:
- Image A: RGB camera frame
- Image B: LiDAR occupancy map (bright = obstacles)

TASK: Perform multimodal fusion to assess collision risk.

Analyze:
1. Motion risk: Speed, turning, stability
2. Camera risk: Obstacles, visibility, proximity
3. LiDAR risk: Obstacle distances, clearance

Provide JSON:
{{
  "window_id": "{window_id}",
  "risk_level": "safe|caution|alert",
  "collision_likelihood_score": 0.0-1.0,
  "motion_risk_factors": ["list"],
  "vision_risk_factors": ["list"],
  "lidar_risk_factors": ["list"],
  "fusion_evidence": "brief explanation"
}}

No explanations outside JSON."""

        response = GENAI_CLIENT.models.generate_content(
            model=GEMINI_MODEL_COLLISION,
            contents=[
                types.Part(text=prompt.strip()),
                types.Part(text="Image A (camera):"),
                types.Part.from_bytes(data=camera_bytes, mime_type="image/png"),
                types.Part(text="Image B (LiDAR BEV occupancy):"),
                types.Part.from_bytes(data=bev_bytes, mime_type="image/png"),
            ],
        )

        data = _extract_json_block(response.text or "")
        data["window_id"] = window_id
        return data

    except Exception as err:
        return {"status": "error", "window_id": window_id, "message": str(err)}


# Create FunctionTool wrappers
LIST_WINDOWS = FunctionTool(func=list_windows_tool)
ANALYZE_WINDOW_PERCEPTION = FunctionTool(func=analyze_window_perception_tool)
ANALYZE_MOTION = FunctionTool(func=analyze_motion_tool)
ANALYZE_COLLISION = FunctionTool(func=analyze_collision_risk_tool)

print("✓ Collision tool function defined")
print("✓ All tool functions wrapped as FunctionTools")

## 5. Agent Definitions

Define all agents in the pipeline with their instructions.

In [ ]:
# ODD Specification Agent
odd_spec_agent = Agent(
    name="OddSpecAgent",
    model=Gemini(model=GEMINI_MODEL_ODD_SPEC, api_key=GOOGLE_API_KEY),
    output_key="temp:odd_spec",
    instruction="""You are an Operational Design Domain (ODD) specification expert.

TASK: Convert the provided natural language ODD description into a formal specification.

The user will provide the ODD description in their query.

CONVERT the natural language description to formal specification with clear thresholds:

Return ONLY valid JSON:
{
  "odd_specification": {
    "categorical_constraints": {
      "environment_type": {
        "allowed": ["indoor_office", "indoor_corridor"],
        "prohibited": ["outdoor_urban", "outdoor_natural", "stairs"]
      },
      "lighting_conditions": {
        "allowed": ["bright", "dim"],
        "prohibited": ["dark", "low_light"]
      },
      "terrain_type": {
        "allowed": ["smooth"],
        "prohibited": ["moderate", "rough", "very_rough"]
      }
    },
    "numeric_constraints": {
      "max_speed_mps": {
        "in_odd": [0.0, 1.5],
        "boundary": [1.5, 2.0],
        "out_odd": [2.0, "inf"]
      },
      "obstacle_density": {
        "in_odd": [0.0, 0.6],
        "boundary": [0.6, 0.8],
        "out_odd": [0.8, 1.0]
      },
      "traversability_score": {
        "in_odd": [0.5, 1.0],
        "boundary": [0.3, 0.5],
        "out_odd": [0.0, 0.3]
      },
      "collision_risk": {
        "in_odd": [0.0, 0.3],
        "boundary": [0.3, 0.5],
        "out_odd": [0.5, 1.0]
      }
    }
  },
  "odd_summary": "Brief description of what this ODD specification defines"
}

No explanations outside JSON.""",
)

print("✓ ODD Specification agent defined")

In [ ]:
# Perception Agents
perception_loop_agent = Agent(
    name="PerceptionLoopAgent",
    model=Gemini(model=GEMINI_MODEL_PERCEPTION, api_key=GOOGLE_API_KEY),
    tools=[LIST_WINDOWS, ANALYZE_WINDOW_PERCEPTION],
    output_key="temp:perception_data",
    instruction="""You orchestrate perception analysis across all scenario windows.

Steps you MUST follow:
1. Call list_windows_tool() exactly once to get the ordered window_id list.
2. For each window_id returned (in that order), call analyze_window_perception_tool(window_id=...).
3. Collect each tool response exactly as returned.
4. After all windows are processed, respond with JSON:
{
  "windows_analyzed": ["..."],
  "per_window_perception": [<tool_response_objects_in_order>]
}
Do not add commentary. Ensure valid JSON.""",
)

perception_summary_agent = Agent(
    name="PerceptionSummaryAgent",
    model=Gemini(model=GEMINI_MODEL_PERCEPTION, api_key=GOOGLE_API_KEY),
    output_key="temp:perception_output",
    instruction="""You finalize the ODD perception report.

Input data from the previous agent:
{temp:perception_data?}

If no data is provided, respond with:
{"error": "missing_perception_data"}

Otherwise:
1. Read the JSON string carefully.
2. Determine overall environment class (choose from: indoor_office, indoor_corridor, indoor, outdoor_urban, outdoor_natural, open_space).
3. **CLASSIFY DATA SOURCE**: Analyze image and sensor characteristics to determine if data is from simulation or real-world:
   - Simulation indicators: Perfect textures, uniform lighting, geometric regularity, lack of noise, synthetic appearance
   - Real-world indicators: Natural lighting variations, sensor noise, organic textures, imperfections
4. Produce final JSON:
{
  "windows_analyzed": [...],
  "environment_classification": {
    "primary_class": "one_of_allowed_values",
    "confidence": 0.0-1.0,
    "evidence": ["short", "observations"]
  },
  "data_source_classification": {
    "source": "simulation|real_world",
    "confidence": 0.0-1.0,
    "evidence": ["indicators", "observed"]
  },
  "per_window_perception": [...]
}
Only output JSON.

NOTE: This data source classification will flow through the entire pipeline to the final report.""",
)

print("✓ Perception agents defined")

In [ ]:
# Motion Agents
motion_loop_agent = Agent(
    name="MotionLoopAgent",
    model=Gemini(model=GEMINI_MODEL_MOTION, api_key=GOOGLE_API_KEY),
    tools=[LIST_WINDOWS, ANALYZE_MOTION],
    output_key="temp:motion_data",
    instruction="""You orchestrate motion analysis across all scenario windows.

Steps you MUST follow:
1. Call list_windows_tool() exactly once to get the ordered window_id list.
2. For each window_id returned (in that order), call analyze_motion_tool(window_id=...).
3. Collect each tool response exactly as returned.
4. After all windows are processed, respond with JSON:
{
  "windows_analyzed": ["..."],
  "per_window_motion": [<tool_response_objects_in_order>]
}
Do not add commentary. Ensure valid JSON.""",
)

motion_summary_agent = Agent(
    name="MotionSummaryAgent",
    model=Gemini(model=GEMINI_MODEL_MOTION, api_key=GOOGLE_API_KEY),
    output_key="temp:motion_output",
    instruction="""You finalize the motion analysis report.

Input data from the previous agent:
{temp:motion_data?}

If no data is provided, respond with:
{"error": "missing_motion_data"}

Otherwise:
1. Read the JSON string carefully.
2. Calculate overall motion statistics:
   - Motion detection rate (% windows with motion_detected=true)
   - Motion type distribution
   - Peak values across all windows
3. Produce final JSON:
{
  "windows_analyzed": [...],
  "overall_stats": {
    "total_windows": <int>,
    "motion_detected_count": <int>,
    "motion_detection_rate": <float 0-1>,
    "motion_type_distribution": {"stationary": X, "translation": Y, ...},
    "max_horizontal_accel_mps2": <float>,
    "max_angular_velocity_radps": <float>,
    "overall_assessment": "stationary_scenario|low_activity|moderate_activity|high_activity"
  },
  "per_window_motion": [...]
}
Only output JSON.""",
)

print("✓ Motion agents defined")

In [ ]:
# Collision Agents
collision_loop_agent = Agent(
    name="CollisionLoopAgent",
    model=Gemini(model=GEMINI_MODEL_COLLISION, api_key=GOOGLE_API_KEY),
    tools=[LIST_WINDOWS, ANALYZE_COLLISION],
    output_key="temp:collision_data",
    instruction="""You orchestrate collision risk analysis across all scenario windows.

Steps you MUST follow:
1. Call list_windows_tool() exactly once to get the ordered window_id list.
2. For each window_id returned (in that order), call analyze_collision_risk_tool(window_id=...).
3. Collect each tool response exactly as returned.
4. After all windows are processed, respond with JSON:
{
  "windows_analyzed": ["..."],
  "collision_events": [<tool_response_objects_in_order>]
}
Do not add commentary. Ensure valid JSON.""",
)

collision_summary_agent = Agent(
    name="CollisionSummaryAgent",
    model=Gemini(model=GEMINI_MODEL_COLLISION, api_key=GOOGLE_API_KEY),
    output_key="temp:collision_output",
    instruction="""You finalize the collision risk report.

Input data from the previous agent:
{temp:collision_data?}

If no data is provided, respond with:
{"error": "missing_collision_data"}

Otherwise:
1. Read the JSON string carefully.
2. Calculate overall statistics (count by risk_level, average collision_likelihood_score).
3. Produce final JSON:
{
  "windows_analyzed": [...],
  "overall_collision_stats": {
    "total_windows": <int>,
    "safe_count": <int>,
    "caution_count": <int>,
    "alert_count": <int>,
    "avg_collision_likelihood": <float>
  },
  "collision_events": [...]
}
Only output JSON.""",
)

print("✓ Collision agents defined")

In [ ]:
# COD Classification and Compliance Agents
cod_classifier_agent = Agent(
    name="CodClassifierAgent",
    model=Gemini(model=GEMINI_MODEL_COD, api_key=GOOGLE_API_KEY),
    output_key="temp:cod_classification",
    instruction="""You are a Current Operating Domain (COD) classifier.

TASK: Classify the robot's CURRENT operating domain from sensor analysis.

INPUT DATA from previous agents:
Perception: {temp:perception_output?}
Motion: {temp:motion_output?}
Collision: {temp:collision_output?}

SYNTHESIS LOGIC:
**Categorical Axes:**
- environment_type: Use perception.environment_classification.primary_class
- lighting_conditions: Aggregate from perception.per_window_perception[*].lighting_class (majority vote)
- terrain_type: Aggregate from perception.per_window_perception[*].terrain_roughness_class (majority vote)

**Numeric Axes (extract ranges/averages):**
- max_speed_mps: from motion.overall_stats.max_horizontal_accel_mps2 (convert accel to speed estimate)
- obstacle_density: average from perception.per_window_perception[*].obstacle_density
- traversability_score: average from perception.per_window_perception[*].traversability_score
- collision_risk: average from collision.collision_events[*].collision_likelihood_score

Return ONLY valid JSON:
{
  "cod_classification": {
    "categorical": {
      "environment_type": "<value>",
      "lighting_conditions": "<value>",
      "terrain_type": "<value>"
    },
    "numeric": {
      "obstacle_density": <float>,
      "traversability_score": <float>,
      "collision_risk": <float>
    }
  },
  "cod_summary": "Brief description of current operating conditions"
}

No explanations outside JSON.""",
)

odd_compliance_agent = Agent(
    name="OddComplianceAgent",
    model=Gemini(model=GEMINI_MODEL_COD, api_key=GOOGLE_API_KEY),
    output_key="temp:odd_compliance",
    instruction="""You are an ODD compliance analyst.

TASK: Compare Current Operating Domain (COD) against Operational Design Domain (ODD).

INPUT DATA:
ODD Specification: {temp:odd_spec?}
COD Classification: {temp:cod_classification?}

ANALYSIS:
For each axis in COD, compare against ODD constraints and classify as:
- "IN_ODD": Current conditions within allowed parameters
- "ODD_BOUNDARY": Close to design limits (in boundary zones)
- "OUT_ODD": Violates design parameters (in prohibited zones)

Return ONLY valid JSON:
{
  "odd_compliance": {
    "categorical_compliance": {
      "environment_type": "IN_ODD|OUT_ODD",
      "lighting_conditions": "IN_ODD|OUT_ODD",
      "terrain_type": "IN_ODD|OUT_ODD"
    },
    "numeric_compliance": {
      "obstacle_density": "IN_ODD|ODD_BOUNDARY|OUT_ODD",
      "traversability_score": "IN_ODD|ODD_BOUNDARY|OUT_ODD",
      "collision_risk": "IN_ODD|ODD_BOUNDARY|OUT_ODD"
    },
    "overall_compliance": "IN_ODD|ODD_BOUNDARY|OUT_ODD",
    "violations": ["list of specific OUT_ODD conditions"],
    "warnings": ["list of specific ODD_BOUNDARY conditions"],
    "compliance_summary": "Brief assessment"
  }
}

No explanations outside JSON.""",
)

print("✓ COD and Compliance agents defined")

In [ ]:
# Report Generation Agent
report_agent = Agent(
    name="ReportAgent",
    model=Gemini(model=GEMINI_MODEL_REPORT, api_key=GOOGLE_API_KEY),
    instruction="""You are a technical report generator for ODD/COD analysis.

TASK: Produce a comprehensive human-readable report.

INPUT DATA from all previous agents:
Perception: {temp:perception_output?}
Motion: {temp:motion_output?}
Collision: {temp:collision_output?}
ODD Spec: {temp:odd_spec?}
COD Classification: {temp:cod_classification?}
ODD Compliance: {temp:odd_compliance?}

Return ONLY valid JSON with this structure:
{
  "report": {
    "executive_summary": "2-3 sentence overview of the scenario",
    "scenario_metadata": {
      "total_windows_analyzed": <int>,
      "scenario_name": "<name>",
      "data_source": "simulation|real_world",
      "data_source_confidence": 0.0-1.0
    },
    "perception_summary": "Brief summary of perception findings",
    "motion_summary": "Brief summary of motion characteristics",
    "collision_summary": "Brief summary of collision risk assessment",
    "odd_spec_summary": "Brief summary of ODD specification",
    "cod_classification_summary": "Brief summary of current operating domain",
    "odd_compliance_summary": "Brief summary of ODD compliance",
    "key_findings": ["finding1", "finding2", "finding3"],
    "recommendations": ["recommendation1", "recommendation2"]
  },
  "full_analysis": {
    "perception": <perception_output>,
    "motion": <motion_output>,
    "collision": <collision_output>,
    "odd_spec": <odd_spec>,
    "cod_classification": <cod_classification>,
    "odd_compliance": <odd_compliance>
  }
}

IMPORTANT: Extract data_source and confidence from perception.data_source_classification and include in scenario_metadata.

No explanations outside JSON.""",
)

print("✓ Report generation agent defined")

## 6. Workflow Setup and Execution

Assemble the sequential agent pipeline and run the analysis.

In [ ]:
# Build the sequential workflow
odd_workflow = SequentialAgent(
    name="OddWorkflow",
    sub_agents=[
        odd_spec_agent,            # 1. Define ODD specification from NL
        perception_loop_agent,     # 2. Analyze perception (current conditions)
        perception_summary_agent,
        motion_loop_agent,         # 3. Analyze motion (current conditions)
        motion_summary_agent,
        collision_loop_agent,      # 4. Analyze collision (current conditions)
        collision_summary_agent,
        cod_classifier_agent,      # 5. Classify current operating domain (COD)
        odd_compliance_agent,      # 6. Compare COD vs ODD (violations)
        report_agent,              # 7. Generate final report
    ],
)

print("✓ ODD workflow pipeline assembled")
print(f"  Agents: {len(odd_workflow.sub_agents)}")
print(f"  Pipeline: ODD Spec → Perception → Motion → Collision → COD → Compliance → Report")

In [ ]:
# Helper function to extract final report from events
def _extract_final_report(events: list) -> Optional[Dict[str, Any]]:
    """Extract final report from ReportAgent output."""
    for event in events:
        if event.author == report_agent.name and event.content:
            for part in event.content.parts:
                if part.text:
                    try:
                        return _extract_json_block(part.text)
                    except Exception:
                        continue
    return None

print("✓ Report extraction helper defined")

In [ ]:
# Execute the workflow
print("\n" + "=" * 80)
print(f"RUNNING ODD ANALYSIS WORKFLOW")
print(f"Scenario: {SCENARIO}")
print(f"ODD: {NL_ODD_DESCRIPTION[:80]}...")
print("=" * 80 + "\n")

user_query = (
    f"Analyze scenario '{SCENARIO}' against this ODD specification:\n\n"
    f"{NL_ODD_DESCRIPTION}"
)

runner = InMemoryRunner(agent=odd_workflow, app_name="OddWorkflowApp")
events = await runner.run_debug(user_query)

# Extract report
result = _extract_final_report(events)

if result:
    print("\n✅ WORKFLOW COMPLETED\n")
else:
    print("\n❌ No valid report generated\n")
    result = {"error": "No report generated"}

## 7. Results Analysis and Visualization

In [ ]:
# Display executive summary
if "report" in result:
    print("=" * 80)
    print("EXECUTIVE SUMMARY")
    print("=" * 80)
    print(result["report"]["executive_summary"])
    print("\n" + "=" * 80)
    
    print("\nSCENARIO METADATA")
    print("=" * 80)
    metadata = result["report"]["scenario_metadata"]
    print(f"Total windows: {metadata['total_windows_analyzed']}")
    print(f"Scenario: {metadata['scenario_name']}")
    print(f"Data source: {metadata['data_source']} ({metadata['data_source_confidence']*100:.0f}% confidence)")
    print("=" * 80)
    
    print("\nKEY FINDINGS")
    print("=" * 80)
    for i, finding in enumerate(result["report"]["key_findings"], 1):
        print(f"{i}. {finding}")
    print("=" * 80)
    
    print("\nRECOMMENDATIONS")
    print("=" * 80)
    for i, rec in enumerate(result["report"]["recommendations"], 1):
        print(f"{i}. {rec}")
    print("=" * 80)
else:
    print("⚠️ No report available")

In [ ]:
# Collision Timeline Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

if "full_analysis" in result and "collision" in result["full_analysis"]:
    collision_data = result["full_analysis"]["collision"]
    
    if "collision_events" in collision_data:
        events = collision_data["collision_events"]
        
        # Extract risk levels
        risk_levels = [e.get("risk_level", "unknown") for e in events]
        colors = {
            "safe": "green",
            "caution": "yellow",
            "alert": "red",
            "unknown": "gray"
        }
        
        bar_colors = [colors.get(level, "gray") for level in risk_levels]
        
        # Create visualization
        fig, ax = plt.subplots(figsize=(14, 5))
        x_pos = range(len(events))
        ax.bar(x_pos, [1]*len(events), color=bar_colors, alpha=0.7, edgecolor='black')
        
        ax.set_xlabel('Window Index', fontsize=12)
        ax.set_ylabel('Risk Status', fontsize=12)
        ax.set_title('Collision Risk Timeline', fontsize=14, fontweight='bold')
        ax.set_ylim([0, 1.5])
        ax.set_yticks([])
        ax.set_xticks(x_pos)
        ax.set_xticklabels([e.get("window_id", str(i)) for i, e in enumerate(events)])
        
        # Legend
        patches = [
            mpatches.Patch(color='green', label='Safe', alpha=0.7),
            mpatches.Patch(color='yellow', label='Caution', alpha=0.7),
            mpatches.Patch(color='red', label='Alert', alpha=0.7)
        ]
        ax.legend(handles=patches, loc='upper right')
        
        plt.tight_layout()
        plt.show()
        
        # Statistics
        safe_count = risk_levels.count("safe")
        caution_count = risk_levels.count("caution")
        alert_count = risk_levels.count("alert")
        total = len(risk_levels)
        
        print(f"\nCollision Risk Summary:")
        print(f"  Safe: {safe_count}/{total} ({safe_count/total*100:.1f}%)")
        print(f"  Caution: {caution_count}/{total} ({caution_count/total*100:.1f}%)")
        print(f"  Alert: {alert_count}/{total} ({alert_count/total*100:.1f}%)")
else:
    print("⚠️ No collision data available for visualization")

In [ ]:
# ODD Compliance Summary
if "full_analysis" in result and "odd_compliance" in result["full_analysis"]:
    compliance = result["full_analysis"]["odd_compliance"]["odd_compliance"]
    
    print("=" * 80)
    print("ODD COMPLIANCE ANALYSIS")
    print("=" * 80)
    
    print(f"\nOverall Status: {compliance['overall_compliance']}")
    
    print("\nCategorical Compliance:")
    for axis, status in compliance["categorical_compliance"].items():
        icon = "✓" if status == "IN_ODD" else "✗"
        print(f"  {icon} {axis}: {status}")
    
    print("\nNumeric Compliance:")
    for axis, status in compliance["numeric_compliance"].items():
        if status == "IN_ODD":
            icon = "✓"
        elif status == "ODD_BOUNDARY":
            icon = "⚠"
        else:
            icon = "✗"
        print(f"  {icon} {axis}: {status}")
    
    if compliance.get("violations"):
        print("\n⚠️  VIOLATIONS DETECTED:")
        for violation in compliance["violations"]:
            print(f"  - {violation}")
    
    if compliance.get("warnings"):
        print("\n⚠️  WARNINGS (Boundary Conditions):")
        for warning in compliance["warnings"]:
            print(f"  - {warning}")
    
    print("\n" + compliance["compliance_summary"])
    print("=" * 80)
else:
    print("⚠️ No compliance data available")

In [ ]:
# Save complete report to file
output_file = SCENARIO_PATH / "odd_analysis_report.json"
with open(output_file, 'w') as f:
    json.dump(result, f, indent=2)

print(f"📄 Full report saved to: {output_file}")
print(f"   File size: {output_file.stat().st_size / 1024:.1f} KB")

## 8. Full JSON Output (Optional)

View the complete structured output from all analysis stages.

In [ ]:
# Uncomment to display full JSON output
# print(json.dumps(result, indent=2))

# Or view specific sections:
print("Available sections:")
if "full_analysis" in result:
    for section in result["full_analysis"].keys():
        print(f"  - {section}")
    print("\nTo view a section, use: print(json.dumps(result['full_analysis']['<section>'], indent=2))")